In [1]:
import os
import sys

In [2]:
current_dir = os.getcwd()
root_dir = os.path.abspath(os.path.join(current_dir, '..'))
sys.path.insert(0, root_dir)

In [3]:
from starter import rag, client, index

In [4]:
query = "How does the agentic loop keep calling the model until it stops?"
answer = rag.rag(query)
print(answer)

# How the Agentic Loop Keeps Calling the Model Until It Stops

Based on the context, the agentic loop uses a **`while True` loop with a simple exit condition** to keep calling the model until it stops making function calls.

## The Core Mechanism

Here's how it works:

```python
while True:
    # Call the model
    response = openai_client.responses.create(
        model="gpt-5.4-mini",
        input=messages,
        tools=[search_tool]
    )

    # Process the response
    messages.extend(response.output)
    has_function_calls = False

    for item in response.output:
        if item.type == "function_call":
            # Execute the function and append results
            call_output = make_call(item)
            messages.append(call_output)
            has_function_calls = True
        elif item.type == "message":
            # Store the final answer
            last_answer = item.content[0].text

    # Exit condition: stop if no function calls this turn
    if has_function_calls 

In [5]:
from opentelemetry import trace
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import ConsoleSpanExporter, SimpleSpanProcessor

In [6]:
class CapturingConsoleExporter(ConsoleSpanExporter):
    def __init__(self):
        super().__init__()
        self.spans = []

    def export(self, spans):
        self.spans.extend(spans)
        return super().export(spans)


console_exporter = CapturingConsoleExporter()

provider = TracerProvider()
provider.add_span_processor(SimpleSpanProcessor(console_exporter))
trace.set_tracer_provider(provider)

tracer = trace.get_tracer("llm-zoomcamp")

In [7]:
from rag_helper import RAGBase
from utils.evaluation_utils import calc_price

## Q1. First trace

In [8]:
class RAGTraced(RAGBase):
    def rag(self, query):
        with tracer.start_as_current_span("rag"):
            return super().rag(query) 
            
    def search(self, query):
        with tracer.start_as_current_span("search") as span:
            results = super().search(query)
            span.set_attribute("num_results", len(results))
            return results
        

    def llm(self, prompt):
        with tracer.start_as_current_span("llm") as span:
            response = super().llm(prompt)
            span.set_attribute("input_tokens", response.usage.input_tokens)
            span.set_attribute("output_tokens", response.usage.output_tokens)
            span.set_attribute("cost", calc_price(response.usage)["total_cost"])
            return response

In [9]:
assistant_rag_traced = RAGTraced(
    index=index,
    llm_client=client
)

In [10]:
query = "How does the agentic loop keep calling the model until it stops?"

In [11]:
results = assistant_rag_traced.rag(query)

{
    "name": "search",
    "context": {
        "trace_id": "0x1b8bd9e22b922d00f4765c4b516a04b2",
        "span_id": "0xd7b74d8e67c2a9b7",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": "0x927905a1a7cf0403",
    "start_time": "2026-07-26T14:51:31.449153Z",
    "end_time": "2026-07-26T14:51:31.456030Z",
    "status": {
        "status_code": "UNSET"
    },
    "attributes": {
        "num_results": 5
    },
    "events": [],
    "links": [],
    "resource": {
        "attributes": {
            "telemetry.sdk.language": "python",
            "telemetry.sdk.name": "opentelemetry",
            "telemetry.sdk.version": "1.39.1",
            "service.name": "unknown_service"
        },
        "schema_url": ""
    }
}
{
    "name": "llm",
    "context": {
        "trace_id": "0x1b8bd9e22b922d00f4765c4b516a04b2",
        "span_id": "0x7995944a87d77f91",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": "0x927905a1a7cf0

In [12]:
results

'# How the Agentic Loop Keeps Calling the Model Until It Stops\n\nBased on the context, the agentic loop uses a **`while True` loop with a flag-based exit condition** to keep calling the model until it stops requesting tools.\n\n## The Core Mechanism\n\nThe loop works like this:\n\n```python\nwhile True:\n    # Call the model\n    response = openai_client.responses.create(\n        model="gpt-5.4-mini",\n        input=messages,\n        tools=[search_tool],\n    )\n    \n    # Add the response to message history\n    messages.extend(response.output)\n    \n    # Check if there are function calls\n    has_function_calls = False\n    \n    for item in response.output:\n        if item.type == "function_call":\n            # Execute the function call\n            call_output = make_call(item)\n            messages.append(call_output)\n            has_function_calls = True\n        elif item.type == "message":\n            # Process the final answer\n            print(item.content[0].text)

In [13]:
spans = console_exporter.spans

In [14]:
len(spans)

3

### There's are 3 spans.

## Q2. Capturing metrics as span attributes

### We have 8183 input tokens

## Q3. Span timing

### Duration of the search span, llm span and rag span

In [15]:
spans_name = ['search', 'llm', 'rag']

In [16]:
for s, n in zip(spans, spans_name):
    span_duration = (s.end_time - s.start_time) / 1_000_000
    print(f"{n} span duration: {span_duration} ms. \n")

search span duration: 6.876702 ms. 

llm span duration: 6677.869301 ms. 

rag span duration: 6689.626062 ms. 



## Q4. Saving traces to SQLite

In [17]:
import sqlite3
from opentelemetry.sdk.trace.export import SpanExporter, SpanExportResult

In [18]:
class SQLiteSpanExporter(SpanExporter):

    def __init__(self, db_path="traces.db"):
        self.conn = sqlite3.connect(db_path)
        self.conn.execute("""
            CREATE TABLE IF NOT EXISTS spans (
                name TEXT,
                start_time INTEGER,
                end_time INTEGER,
                input_tokens INTEGER,
                output_tokens INTEGER,
                cost REAL
            )
        """)
        self.conn.commit()

    def export(self, spans):
        for span in spans:
            attrs = dict(span.attributes or {})
            self.conn.execute(
                "INSERT INTO spans VALUES (?, ?, ?, ?, ?, ?)",
                (
                    span.name,
                    span.start_time,
                    span.end_time,
                    attrs.get("input_tokens"),
                    attrs.get("output_tokens"),
                    attrs.get("cost"),
                ),
            )
        self.conn.commit()
        return SpanExportResult.SUCCESS

    def shutdown(self):
        self.conn.close()

    def force_flush(self):
        return True

In [19]:
provider.add_span_processor(
    SimpleSpanProcessor(SQLiteSpanExporter("traces.db"))
)

In [20]:
results = assistant_rag_traced.rag(query)

{
    "name": "search",
    "context": {
        "trace_id": "0x8e6c137b0c761d615431b1ec90d804fa",
        "span_id": "0x8f2d634f1726eac8",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": "0x5f0ade50607b9d99",
    "start_time": "2026-07-26T14:52:50.231023Z",
    "end_time": "2026-07-26T14:52:50.237262Z",
    "status": {
        "status_code": "UNSET"
    },
    "attributes": {
        "num_results": 5
    },
    "events": [],
    "links": [],
    "resource": {
        "attributes": {
            "telemetry.sdk.language": "python",
            "telemetry.sdk.name": "opentelemetry",
            "telemetry.sdk.version": "1.39.1",
            "service.name": "unknown_service"
        },
        "schema_url": ""
    }
}
{
    "name": "llm",
    "context": {
        "trace_id": "0x8e6c137b0c761d615431b1ec90d804fa",
        "span_id": "0x8f3762a17319bc02",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": "0x5f0ade50607b9

## Q5. Querying trace data

In [21]:
query = "How can I have a certificate ?"
results = assistant_rag_traced.rag(query)

{
    "name": "search",
    "context": {
        "trace_id": "0x8d4a13c335d1b5681ca03d013a246346",
        "span_id": "0xf5966afa546a6177",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": "0x27ee0d16a92a0f9a",
    "start_time": "2026-07-26T14:53:03.883682Z",
    "end_time": "2026-07-26T14:53:03.890714Z",
    "status": {
        "status_code": "UNSET"
    },
    "attributes": {
        "num_results": 5
    },
    "events": [],
    "links": [],
    "resource": {
        "attributes": {
            "telemetry.sdk.language": "python",
            "telemetry.sdk.name": "opentelemetry",
            "telemetry.sdk.version": "1.39.1",
            "service.name": "unknown_service"
        },
        "schema_url": ""
    }
}
{
    "name": "llm",
    "context": {
        "trace_id": "0x8d4a13c335d1b5681ca03d013a246346",
        "span_id": "0x245f11235aa33c02",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": "0x27ee0d16a92a0

In [22]:
import pandas as pd

In [35]:
df = pd.read_sql("SELECT * FROM spans;", sqlite3.connect("traces.db"))
df

,name,start_time,end_time,input_tokens,output_tokens,cost
0,search,1785077570231023192,1785077570237261488,NaN,NaN,NaN
1,llm,1785077570242767293,1785077575948841715,8183.0,419.0,0.010278
2,rag,1785077570230926248,1785077575952391718,NaN,NaN,NaN
3,search,1785077583883681730,1785077583890713772,NaN,NaN,NaN
4,llm,1785077583896813042,1785077586200464433,5631.0,127.0,0.006266
5,rag,1785077583883577646,1785077586206907474,NaN,NaN,NaN
6,search,1785077674741677681,1785077674747478662,NaN,NaN,NaN
7,llm,1785077674754347409,1785077680613366939,8183.0,483.0,0.010598
8,rag,1785077674741584505,1785077680629281835,NaN,NaN,NaN
9,search,1785077699573298170,1785077699579110056,NaN,NaN,NaN


In [32]:
df['start_time_ms'] = df['start_time'].apply(lambda x: x / 1_000_000)
df['end_time_ms'] = df['end_time'].apply(lambda x: x / 1_000_000)

In [33]:
df['total_duration_ms'] = df['end_time_ms'] - df['start_time_ms']

In [34]:
df[['name', 'total_duration_ms']].groupby('name')['total_duration_ms'].mean()

name
llm       4927.939502
rag       4947.805420
search       6.084375
Name: total_duration_ms, dtype: float64

## Q6. Token stability across runs

In [29]:
query = "How does the agentic loop keep calling the model until it stops?"
assistant_rag_traced.rag(query)

{
    "name": "search",
    "context": {
        "trace_id": "0x6007b7ad7bf31ab2456f1b87fc5f991e",
        "span_id": "0x5f13a1093aaa6f9d",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": "0x8d175faef5ba9f04",
    "start_time": "2026-07-26T14:55:08.791536Z",
    "end_time": "2026-07-26T14:55:08.797075Z",
    "status": {
        "status_code": "UNSET"
    },
    "attributes": {
        "num_results": 5
    },
    "events": [],
    "links": [],
    "resource": {
        "attributes": {
            "telemetry.sdk.language": "python",
            "telemetry.sdk.name": "opentelemetry",
            "telemetry.sdk.version": "1.39.1",
            "service.name": "unknown_service"
        },
        "schema_url": ""
    }
}
{
    "name": "llm",
    "context": {
        "trace_id": "0x6007b7ad7bf31ab2456f1b87fc5f991e",
        "span_id": "0x18e69f5e1eed13fb",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": "0x8d175faef5ba9

'# How the Agentic Loop Keeps Calling the Model Until It Stops\n\nBased on the context, the agentic loop uses a **simple flag-based exit condition** to keep calling the model until it stops:\n\n## The Core Mechanism\n\nThe loop maintains a `has_function_calls` flag that tracks whether the model\'s response contains any function calls:\n\n```python\nwhile True:\n    print(f"iteration #{it}...")\n    has_function_calls = False\n\n    response = openai_client.responses.create(\n        model="gpt-5.4-mini",\n        input=messages,\n        tools=[search_tool],\n    )\n\n    messages.extend(response.output)\n\n    for item in response.output:\n        if item.type == "function_call":\n            # Execute the function call\n            has_function_calls = True\n        elif item.type == "message":\n            # Process the message\n            pass\n\n    it = it + 1\n    if has_function_calls == False:\n        break\n```\n\n## How It Works\n\n1. **Each iteration**: The model is calle

### The input tokens don't vary across these 2 runs